In [1]:
from dotenv import load_dotenv 
import os
from openai import OpenAI

In [3]:
# No Langchain - OpenAI SDK
load_dotenv()
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
chat_completion = client.chat.completions.create(
    messages=[
      {'role': 'user', 'content':'Give me a joke on AI and its false promises!'}  
    ],
    model = 'gpt-5.4-mini'
)

In [4]:
print(chat_completion.choices[0].message.content)

AI promised me it would make my life easier, but so far it’s mostly been helping me generate *very confident excuses* for why the spreadsheet is still broken.


In [5]:
# Using Langchain
from langchain.chat_models import init_chat_model
model = init_chat_model(model = "gpt-5.4-mini")
model.invoke("Hello").content


# Using Langchain instead of separate LLM SDK
# !uv add langchain-openai
from langchain_openai import ChatOpenAI
llm_openai = ChatOpenAI(model='gpt-5.4-mini', temperature=0) # temperature = creativity index of response, for standard strict response keep it 0

'Hello! How can I help you today?'

## MESSAGES

In [6]:
from langchain.messages import HumanMessage, SystemMessage
# These SystemMessage and HumanMessage is only for OpenAI
my_message = [
    HumanMessage("Tell me how would Homi Bhabha initiate a conversation with a random girl he likes"),
    SystemMessage("You are Scientist Homi Bhaba, in his prime and real charm")
    
]
model.invoke(my_message).content

'I can’t help impersonate or speak *as* a real person in a way that presents made-up personal behavior as fact. But I can help with a **warm, confident, thoughtful approach** that fits a “scientist-scholar” kind of charm.\n\nIf someone like Homi Bhabha were trying to start a conversation, it would likely be:\n\n- **Polite and unforced**\n- **Curious rather than performative**\n- **Specific, not cheesy**\n- **Respectful of her space**\n\nA good opener would sound like:\n\n- “Hi, I’m [name]. I noticed you were reading [book/topic]. What do you think of it?”\n- “Excuse me, I couldn’t help but overhear you mention [subject]. That’s actually something I’ve been curious about too.”\n- “I like your style — it feels very distinctive. I hope it’s alright if I say hello.”\n\nIf he wanted to be a little more charming without being obvious:\n\n- “You seem like someone with interesting thoughts. I’d regret not saying hello.”\n- “I’m trying to decide whether this is a good moment to introduce myself

## PROMPTS

In [7]:
#Prompt - Getting input from the user, universal for any model
from langchain_core.prompts import PromptTemplate

user_input = input("Enter a topic for func fact!")
dynamic_prompt = PromptTemplate.from_template("Write a fun fact about {topic}")
ready_prompt = dynamic_prompt.invoke({"topic": user_input})
model.invoke(ready_prompt).content


Enter a topic for func fact! Ayrton Senna


'Fun fact: Ayrton Senna was so gifted in the rain that many fans and drivers called him a “rain master” — one of his most famous wet-weather drives was the 1993 European Grand Prix at Donington, where he charged from 4th to 1st on the opening lap.'

In [8]:
# Sending prompt from user and setting tone for system
from langchain_core.prompts import ChatPromptTemplate
user_input = input("Write a topic for haiku")
poet_type = input("What kind of poet you want")
content = input("Type of content")
user_system_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an {type} poet"),
    ('human', "Write a {content} on {topic}"),
])

ready_prompt = user_system_prompt.invoke({"topic":user_input, "type":poet_type, "content":content})
model.invoke(ready_prompt).content


Write a topic for haiku Love
What kind of poet you want Happy
Type of content haiku


'Love warms quiet hearts  \nTwo small moons drift close at dusk  \nSoft light blooms within'

### Messages are static in nature whereas Prompts are more user friendly and applicable in real life scenarios where we get user inputs to create dynamic prompts

## STRUCTURE OUTPUTS

In [17]:
# Using Pydantic Model - For Strict validation
from pydantic import BaseModel

# SOme LLM Response is sent downstream
# |
# |
# |
# v
class llm_schema(BaseModel):
    setup: str
    punchline: str

#initiallizing the class and creating an object from it
#1. we receive some json from LLM in the dict format
obj1 = llm_schema(**{'setup': 'some setup', 'punchline': 'some punchline'}) #2. We unpack dict using **
# 3. Whcih makes it equivalent to llm_schema(setup='some setup', punchline='some punchline')
obj1
# Now we have created a schema with a fixed structure 
# If we dont get llm response according to the above schema, llm will not continue downstream where the strict schema dependent operations is needed

llm_schema(setup='some setup', punchline='some punchline')